# Repeated Split Robustness on Google Colab GPU

This notebook reruns the final wafer follow-up policies across repeated train/validation/test splits.

Important: select `Runtime > Change runtime type > GPU` before running.

Protocol:

- retrain sparse CNN per seed
- retrain non-CNN point-risk model per seed
- evaluate `coverage32`, `noncnn_top32`, `cnn_top32`, and `ensemble_raw_w0.30`
- save CSV/model/report outputs to Google Drive

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure Paths

Use one project source:

- Drive folder mode: copy the local project folder to Google Drive first.
- GitHub mode: clone the repo. This still needs the dataset in Drive.

In [ ]:
from pathlib import Path
import os

use_git_clone = False
GITHUB_REPO_URL = 'https://github.com/JUYONG-1010/wafer-risk-followup-sampling.git'

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/wafer-risk-followup-sampling')
DRIVE_PROJECT_ZIP = Path('/content/drive/MyDrive/wafer-risk-followup-sampling-colab.zip')
DRIVE_LSWMD_PATH = Path('/content/drive/MyDrive/wafer_project_data/LSWMD.pkl')

WORK_DIR = Path('/content/wafer-risk-followup-sampling')
RESULTS_DIR = Path('/content/drive/MyDrive/wafer_project_results/repeated_split_robustness_colab_v1')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('WORK_DIR:', WORK_DIR)
print('RESULTS_DIR:', RESULTS_DIR)
print('DRIVE_PROJECT_DIR exists:', DRIVE_PROJECT_DIR.exists())
print('DRIVE_PROJECT_ZIP exists:', DRIVE_PROJECT_ZIP.exists())
print('DRIVE_LSWMD_PATH exists:', DRIVE_LSWMD_PATH.exists())

## 3. Get Project Code

In [ ]:
import shutil
import subprocess

if WORK_DIR.exists():
    print('Project already exists at', WORK_DIR)
else:
    if use_git_clone:
        subprocess.run(['git', 'clone', GITHUB_REPO_URL, str(WORK_DIR)], check=True)
    else:
        if DRIVE_PROJECT_DIR.exists():
            shutil.copytree(DRIVE_PROJECT_DIR, WORK_DIR)
        elif DRIVE_PROJECT_ZIP.exists():
            WORK_DIR.mkdir(parents=True, exist_ok=True)
            shutil.unpack_archive(str(DRIVE_PROJECT_ZIP), str(WORK_DIR))
        else:
            raise FileNotFoundError(f'No project folder or zip found in Drive: {DRIVE_PROJECT_DIR}, {DRIVE_PROJECT_ZIP}')

# Repair archives created on Windows if backslashes became literal filename characters.
moved = 0
for p in list(WORK_DIR.iterdir()):
    if '\\' in p.name:
        dst = WORK_DIR.joinpath(*p.name.split('\\'))
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            if dst.is_dir():
                shutil.rmtree(dst)
            else:
                dst.unlink()
        shutil.move(str(p), str(dst))
        moved += 1
if moved:
    print('repaired flat Windows zip paths:', moved)

os.chdir(WORK_DIR)
print('cwd:', Path.cwd())
print('top-level files:', sorted([p.name for p in WORK_DIR.iterdir()])[:30])

## 4. Install Dependencies And Check GPU

Do not install `requirements-cnn.txt` in Colab because it pins CPU-only PyTorch. Colab GPU runtimes normally include CUDA PyTorch already.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU is not available. Select Runtime > Change runtime type > GPU and rerun.')

## 5. Prepare Dataset

The robustness runner needs `data/processed/subsets/patterned_subset.pkl`. If the processed subset is missing from the project copy, this cell copies `LSWMD.pkl` from Drive and runs the extraction script.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

patterned_subset = WORK_DIR / 'data/processed/subsets/patterned_subset.pkl'
if patterned_subset.exists():
    print('Found patterned subset:', patterned_subset)
else:
    if not DRIVE_LSWMD_PATH.exists():
        raise FileNotFoundError(f'Missing Drive dataset: {DRIVE_LSWMD_PATH}')
    local_data_dir = WORK_DIR / 'LSWMD.pkl'
    local_data_dir.mkdir(parents=True, exist_ok=True)
    local_pickle = local_data_dir / 'LSWMD.pkl'
    if not local_pickle.exists():
        shutil.copy2(DRIVE_LSWMD_PATH, local_pickle)
    print('Running labeled subset extraction...')
    subprocess.run([sys.executable, 'experiments/01_extract_labeled_subset.py'], check=True)
    if not patterned_subset.exists():
        raise FileNotFoundError(f'Extraction did not create: {patterned_subset}')
print('patterned subset size MB:', round(patterned_subset.stat().st_size / 1024 / 1024, 2))

## 6. Run Repeated Split Robustness

Recommended first run:

- seeds: `42 101 202`
- epochs: `8`
- max train wafers: `2500`
- max test wafers: `500`

This is the leakage-safe version: CNN and non-CNN are retrained separately per seed.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, 'experiments/79_run_repeated_split_robustness_colab.py',
    '--patterned', str(patterned_subset),
    '--output-root', str(RESULTS_DIR / 'data'),
    '--figure-root', str(RESULTS_DIR / 'figures'),
    '--model-root', str(RESULTS_DIR / 'models'),
    '--seeds', '42', '101', '202',
    '--densities', '0.01', '0.03', '0.05', '0.10',
    '--top-k', '32',
    '--epochs', '8',
    '--patience', '3',
    '--max-train-wafers', '2500',
    '--max-test-wafers', '500',
    '--batch-size', '64',
    '--hidden-channels', '32',
    '--point-estimators', '100',
    '--threads', '4',
    '--use-amp',
    '--skip-existing',
]
print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## 7. Inspect Aggregated Results

In [ ]:
import pandas as pd

summary_path = RESULTS_DIR / 'data/repeated_split_policy_robustness_summary.csv'
pattern_path = RESULTS_DIR / 'data/repeated_split_pattern_robustness_summary.csv'
report_path = RESULTS_DIR / 'data/repeated_split_robustness_report.md'

summary = pd.read_csv(summary_path)
display(summary[[
    'target_density', 'final_policy',
    'mean_followup_precision_at_k_mean', 'mean_followup_precision_at_k_std',
    'mean_defect_coverage_mean', 'mean_defect_coverage_std',
    'mean_followup_defects_mean', 'mean_followup_defects_std',
]].round(5))

print('report:', report_path)
print('pattern summary:', pattern_path)